# Lab 7 — ReAct from scratch: the two-node loop, and what `temperature` does to it

So far we've built specialised LangGraph workflows (`planning`, `self_debug`,
multi-step `deep_research`). This lab steps back to the classic **ReAct** pattern
— Reasoning + Acting — as a *two-node* graph: an `agent` node that thinks and
optionally calls tools, a `tools` node that runs them, and a conditional edge
that loops between them until the model is done.

We deliberately build the graph from scratch — not via `langgraph.prebuilt.create_react_agent`
— so the loop is visible in the graph itself, not hidden behind a helper.

This notebook mirrors `cmbagent_lg/examples/react_demo.py` cell-for-cell; the
script is the runnable headless version, this is the live walkthrough.

**The pedagogical twist:** we use a question the model has *already memorised*
("What is the population of France?") to make the `temperature` knob do
something visible. At T=0 the model dutifully calls the lookup tool every
time; at T=1 it's tempted to answer from priors and the tool-call rate drops.


## Prerequisites

Same setup as Labs 5/6 — no `[escalation]` extra needed this time:

- `pip install -e '~/GitHub/cmbagent_lg'` (plain install is enough)
- `GOOGLE_API_KEY` set in `~/GitHub/cmbagent_lg/.env`
- (optional) `LANGFUSE_*` in the same `.env` for tracing

Pick the **Python 3.12 (cmbagent_lg)** kernel.


In [ ]:
from dotenv import load_dotenv

load_dotenv('/Users/boris/GitHub/cmbagent_lg/.env', override=True)

import os
assert os.environ.get('GOOGLE_API_KEY'), 'missing GOOGLE_API_KEY in .env'

# Langfuse is optional — attach the handler if its keys are present.
callbacks, handler = [], None
try:
    from cmbagent_lg.tracing import langfuse_handler
    handler = langfuse_handler()
    callbacks = [handler]
    print('env OK — Langfuse tracing attached')
except Exception as e:
    print(f'env OK — running without Langfuse ({e})')


## 1. The tool — deliberately non-canonical numbers

One tool: `lookup_population`. The numbers don't match Wikipedia — that's the
point. If the agent ever answers with `67,000,000` for France we know the
tool fired; if it answers `~68 million` (the canonical figure) we know the
model is reciting from priors.


In [ ]:
from langchain_core.tools import tool

POPULATIONS = {
    'France':  67_000_000,
    'Germany': 84_000_000,
    'Italy':   59_000_000,
}

@tool
def lookup_population(country: str) -> str:
    """Look up a country's population in a small internal reference table."""
    if country in POPULATIONS:
        return f'The population of {country} is {POPULATIONS[country]:,}.'
    return f'No data for {country}.'

print(lookup_population.invoke({'country': 'France'}))
print(lookup_population.invoke({'country': 'Atlantis'}))


## 2. The state — a TypedDict with `add_messages`

ReAct's whole memory is the running message list. `add_messages` is the
LangGraph reducer that *appends* to the list as nodes return new messages
(rather than overwriting), so the agent sees the full Thought / Action /
Observation history on every turn.


In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


## 3. The graph — two nodes, one conditional edge

```
START ──▶ agent ──┐
          ▲       │ tool_calls?
          │       ▼
          └── tools ──── (if tool_calls present)
                  │
                  └──▶ END (if not)
```

- **`agent_node`** invokes the model with the current message history. Either
  it returns a final answer (no `tool_calls`) or it asks for one or more tool
  calls.
- **`tool_node`** runs every requested tool call and appends a `ToolMessage`
  per call.
- **`should_continue`** is the conditional edge — route to `tools` if the
  last message has `tool_calls`, otherwise to `END`.

`make_graph(temperature)` returns a compiled graph parameterised by the
sampling temperature.


In [ ]:
from langchain_core.messages import ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph


def make_graph(temperature: float):
    model = ChatGoogleGenerativeAI(
        model=os.environ.get('REACT_MODEL', 'gemini-3.1-flash-lite'),
        temperature=temperature,
        thinking_level='low',
        google_api_key=os.environ['GOOGLE_API_KEY'],
    ).bind_tools([lookup_population])

    def agent_node(state: State):
        return {'messages': [model.invoke(state['messages'])]}

    def tool_node(state: State):
        last = state['messages'][-1]
        outs = []
        for tc in last.tool_calls:
            result = lookup_population.invoke(tc['args'])
            outs.append(ToolMessage(content=result, tool_call_id=tc['id']))
        return {'messages': outs}

    def should_continue(state: State):
        last = state['messages'][-1]
        return 'tools' if getattr(last, 'tool_calls', None) else END

    g = StateGraph(State)
    g.add_node('agent', agent_node)
    g.add_node('tools', tool_node)
    g.add_edge(START, 'agent')
    g.add_conditional_edges('agent', should_continue, {'tools': 'tools', END: END})
    g.add_edge('tools', 'agent')
    return g.compile()


graph_cold = make_graph(temperature=0.0)


### The shape, visualised


In [ ]:
from IPython.display import Image, display

g = graph_cold.get_graph()
try:
    display(Image(g.draw_mermaid_png()))
except Exception:
    try:
        print(g.draw_ascii())
    except Exception:
        print('nodes:', [n.id for n in g.nodes.values()])
        print('edges:', [(e.source, e.target) for e in g.edges])


## 4. The system prompt — *permission* to skip the tool

The headline experiment depends on a subtle prompt choice: we tell the model
it *doesn't have to* use the tool. Without that, even a hot model dutifully
calls `lookup_population` every time, and the temperature knob does nothing
visible.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from cmbagent_lg.prompt_utils import flatten_content

SYSTEM_PROMPT = (
    "You are a knowledgeable assistant. You have access to a population "
    "lookup tool, but you don't have to use it — if you already know the "
    "answer with reasonable confidence, just answer directly. Only call "
    "the tool when you're genuinely uncertain."
)


def run_once(graph, question: str, label: str = 'run') -> dict:
    config = {
        'callbacks': callbacks,
        'run_name': f'react_demo · {label}',
        'tags': ['lab7', 'react_demo', label],
    }
    final = graph.invoke(
        {'messages': [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=question),
        ]},
        config=config,
    )
    msgs = final['messages']
    tool_called = any(getattr(m, 'tool_calls', None) for m in msgs)
    final_text = flatten_content(getattr(msgs[-1], 'content', ''))
    return {'tool_called': tool_called, 'final_text': final_text, 'messages': msgs}


def print_trace(messages):
    """Pretty-print the full ReAct trace of one run — for live demos."""
    for i, m in enumerate(messages):
        role = m.__class__.__name__.replace('Message', '')
        body = flatten_content(getattr(m, 'content', '')).strip()
        tcs = getattr(m, 'tool_calls', None) or []
        print(f'\n[{i}] {role}')
        if body:
            print(f'    {body}')
        for tc in tcs:
            print(f"    → tool_call: {tc['name']}({tc['args']})")


## 5. One full trace at T=0 — the cold case

At temperature 0 the model takes the prompt at face value, sees a relevant
tool, and uses it. Expect four messages: `System → Human → AI(tool_call) → Tool → AI(final)`.


In [ ]:
QUESTION = 'What is the population of France?'

print(f'=== Trace at T=0.0 ===\nQ: {QUESTION}')
out_cold = run_once(graph_cold, QUESTION, label='T=0.0 · trace')
print_trace(out_cold['messages'])


## 6. Now T=1 — same question, hotter model

Recompile the graph with `temperature=1.0` and ask the same thing. The model
is more likely to skip the tool and answer from priors. You may need to run
the cell a couple of times to see the contrast — at T=1 the path is genuinely
stochastic.


In [ ]:
graph_hot = make_graph(temperature=1.0)

print(f'=== Trace at T=1.0 ===\nQ: {QUESTION}')
out_hot = run_once(graph_hot, QUESTION, label='T=1.0 · trace')
print_trace(out_hot['messages'])


## 7. Tool-call rate vs temperature — the headline

One trace is a coin flip; the effect shows up in aggregate. Run each graph
`N` times at the same question and count how often the tool fired.

Expect roughly: **T=0 → ~100% tool-call rate, T=1 → noticeably lower.**
(Numbers move run to run — that's the whole point of `temperature`.)


In [ ]:
N = 8

def measure(graph, label: str) -> float:
    hits = 0
    for i in range(N):
        out = run_once(graph, QUESTION, label=f'{label} · run {i+1}/{N}')
        hits += int(out['tool_called'])
        flag = 'TOOL' if out['tool_called'] else 'SKIP'
        snippet = out['final_text'].replace('\n', ' ').strip()[:100]
        print(f'  [{label}] run {i+1:>2}  [{flag}]  {snippet}')
    rate = hits / N
    print(f'  [{label}] tool-call rate: {hits}/{N} = {rate:.0%}\n')
    return rate

rate_cold = measure(graph_cold, 'T=0.0')
rate_hot  = measure(graph_hot,  'T=1.0')

print(f'\n=== Summary ===')
print(f'  T=0.0  tool-call rate: {rate_cold:.0%}')
print(f'  T=1.0  tool-call rate: {rate_hot:.0%}')
print(f'  delta: {rate_cold - rate_hot:+.0%}')


## 8. When the model *can't* fake it — uncertainty forces the tool

The temperature lever only works because France is in the model's priors.
Ask about a country it has no memorised answer for, and even the hot model
falls back to the tool — or, since our table has no entry, the tool truthfully
says it doesn't know either.


In [ ]:
UNKNOWN_Q = 'What is the population of Atlantis?'

print(f'=== T=1.0 on a question with no priors ===\nQ: {UNKNOWN_Q}')
out_unknown = run_once(graph_hot, UNKNOWN_Q, label='T=1.0 · atlantis')
print_trace(out_unknown['messages'])


## Recap

- **ReAct** is just an `agent ↔ tools` loop with a conditional edge — two
  nodes, one router, one state field. Worth seeing built from scratch once
  before reaching for `create_react_agent`.
- **`bind_tools`** gives the model the schema; whether it *uses* the tool is
  a runtime decision shaped by the system prompt, the question, *and the
  temperature*.
- **`temperature`** is not just "creativity for prose" — it changes
  acting decisions inside an agent loop. Worth a knob in any agent demo
  that has a memorised-answer escape hatch.
- The same logic, headless and reproducible, lives in
  `cmbagent_lg/examples/react_demo.py` — run that with `--show-trace` or
  `--repeat N` for the CLI version of this notebook.
